# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring a dataset using the `mlcroissant` library, referencing all dataset entities by their Croissant `@id`s.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Not a dict, but an mlcroissant.Metadata object

print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets and field IDs. All entities are referenced by their Croissant `@id`.

Let's list the available record sets and their fields.

In [ ]:
# List all record sets in the dataset using their @id
record_sets = metadata.record_sets
print(f"Total record sets: {len(record_sets)}\n")

for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"  @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Name: {field.name}")
        print(f"      @id: {field.id}")
        print(f"      Type: {field.data_type}")
    print('')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

We'll extract tabular data from the primary protein analysis record set.

In [ ]:
# Choose the record set to extract data from (by @id)
# For this dataset, let's list them and then pick the main proteins table. If unsure, use the first one as an example.
main_record_set_id = record_sets[0].id if record_sets else None  # Replace with correct id if needed
print(f"Using record set: {main_record_set_id}")

# Extract all record sets into DataFrames for further analysis
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df

if main_record_set_id and main_record_set_id in dataframes:
    print('Columns in the main record set:')
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print('No tabular data found for the selected record set.')

## 4. Exploratory Data Analysis (EDA)
We will process quantitative fields by their Croissant field `@id` and demonstrate filtering, normalization, and grouping.

**Note:** Update the field `@id`s below according to the overview above if running interactively.

In [ ]:
# Pick a numeric field (e.g., peptide count, coverage, or abundance field by @id). Use an overview cell above to find options.
# Example: suppose the field @id is 'protein-abundance' (replace with actual id if different)
numeric_field_id = None
for field in record_sets[0].fields:
    if field.data_type in ('Float', 'Integer', 'Number'):
        numeric_field_id = field.id
        print(f"Selected numeric field: {field.name} (@id: {numeric_field_id})")
        break

# Defensive: only proceed if numeric_field_id is known
if main_record_set_id and numeric_field_id and main_record_set_id in dataframes and numeric_field_id in dataframes[main_record_set_id].columns:
    df = dataframes[main_record_set_id].copy()
    numeric_col = numeric_field_id
    df[numeric_col] = pd.to_numeric(df[numeric_col], errors='coerce')

    threshold = df[numeric_col].quantile(0.75)  # Example: top 25% as 'high abundance'
    filtered_df = df[df[numeric_col] > threshold]
    print(f"Filtered records with {numeric_col} > {threshold:.2f}:")
    display(filtered_df.head())

    filtered_df[f"{numeric_col}_normalized"] = (filtered_df[numeric_col] - filtered_df[numeric_col].mean()) / filtered_df[numeric_col].std()
    print(f"\nNormalized {numeric_col} for filtered records:")
    display(filtered_df[[numeric_col, f"{numeric_col}_normalized"]].head())

    # Find a group field (e.g., by description or sample/condition id)
    group_field_id = None
    for field in record_sets[0].fields:
        if field.data_type in ('Text', 'String') and field.id != numeric_field_id:
            group_field_id = field.id
            print(f'Using group field: {group_field_id}')
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_col].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_col}):")
        display(grouped_df.head())
else:
    print('No valid numeric field found for EDA.')

## 5. Visualization
Visualize data distributions and relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check we have numeric_field_id and data
if main_record_set_id and numeric_field_id and main_record_set_id in dataframes and numeric_field_id in dataframes[main_record_set_id].columns:
    df = dataframes[main_record_set_id]
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If we have a grouping field from above, plot mean by group
    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10, 4))
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        sns.barplot(x=grouped.index, y=grouped.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print('Data not available for visualization.')

## 6. Conclusion
In this notebook, we demonstrated loading a Croissant-based mass spectrometry dataset, explored schema structure using entity `@id`s, extracted tabular data by record set, performed data filtering and normalization, and visualized protein quantitation fields. This workflow can be adapted for FAIR-compliant datasets referencing fields solely by Croissant `@id`.